Here, we will load the student model using FP16 instead of the original FP32 and then also use int8 and int4 and compare the validation loss on all of them


In [ ]:
%pip install -q transformers datasets evaluate bitsandbytes accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 18.8 MB/s eta 0:00:00


In [ ]:
import os
import math
from typing import Dict, Tuple

import torch
from torch.utils.data import DataLoader


from datasets import load_dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    BitsAndBytesConfig
)


In [ ]:
from sklearn.metrics import accuracy_score, f1_score

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

Config: model & dataset IDs


In [ ]:
# Your distilled student model on the Hub (from Notebook 3)
STUDENT_MODEL_ID = "saketgarodia1/bert-it-ticket-student"

# Your IT ticket dataset
DATASET_ID = "saketgarodia1/IT-service-topic-classification-data"

# Max sequence length for tokenization
MAX_LENGTH = 256

# Batch sizes for evaluation
VAL_BATCH_SIZE = 32
TEST_BATCH_SIZE = 32


 Load dataset & build validation / test DataLoaders


In [ ]:
ds = load_dataset(DATASET_ID)
ds

README.md:   0%|          | 0.00/537 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/4.78M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/1.03M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/1.05M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/33485 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/7176 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7176 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['Document', 'Topic_group'],
        num_rows: 33485
    })
    validation: Dataset({
        features: ['Document', 'Topic_group'],
        num_rows: 7176
    })
    test: Dataset({
        features: ['Document', 'Topic_group'],
        num_rows: 7176
    })
})

In [ ]:
classes = sorted(ds["train"].unique("Topic_group"))
classes

['Access',
 'Administrative rights',
 'HR Support',
 'Hardware',
 'Internal Project',
 'Miscellaneous',
 'Purchase',
 'Storage']

In [ ]:
id2label = {id:label for id, label in enumerate(classes)}
id2label

{0: 'Access',
 1: 'Administrative rights',
 2: 'HR Support',
 3: 'Hardware',
 4: 'Internal Project',
 5: 'Miscellaneous',
 6: 'Purchase',
 7: 'Storage'}

In [ ]:
label2id = {label:id for id, label in enumerate(classes)}
label2id

{'Access': 0,
 'Administrative rights': 1,
 'HR Support': 2,
 'Hardware': 3,
 'Internal Project': 4,
 'Miscellaneous': 5,
 'Purchase': 6,
 'Storage': 7}

In [ ]:
label2id['Storage']

7

In [ ]:
def encode_labels(example):
    example["label"] = label2id[example["Topic_group"]]
    return example

ds = ds.map(encode_labels)
ds

Map:   0%|          | 0/33485 [00:00<?, ? examples/s]

Map:   0%|          | 0/7176 [00:00<?, ? examples/s]

Map:   0%|          | 0/7176 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['Document', 'Topic_group', 'label'],
        num_rows: 33485
    })
    validation: Dataset({
        features: ['Document', 'Topic_group', 'label'],
        num_rows: 7176
    })
    test: Dataset({
        features: ['Document', 'Topic_group', 'label'],
        num_rows: 7176
    })
})

In [ ]:
# Load the *same* tokenizer you trained with
tokenizer_student = AutoTokenizer.from_pretrained(STUDENT_MODEL_ID)

# Load the distilled student classifier
model_student = AutoModelForSequenceClassification.from_pretrained(STUDENT_MODEL_ID)
model_student.to(device)
model_student.eval()

DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


In [ ]:
ds

DatasetDict({
    train: Dataset({
        features: ['Document', 'Topic_group', 'label'],
        num_rows: 33485
    })
    validation: Dataset({
        features: ['Document', 'Topic_group', 'label'],
        num_rows: 7176
    })
    test: Dataset({
        features: ['Document', 'Topic_group', 'label'],
        num_rows: 7176
    })
})

In [ ]:
def tokenize_fn(batch):
    return tokenizer_student(
        batch["Document"],
        truncation=True,
        padding = False,
        max_length=MAX_LENGTH,
    )

ds_tok = ds.map(tokenize_fn, batched=True)
ds_tok


Map:   0%|          | 0/33485 [00:00<?, ? examples/s]

Map:   0%|          | 0/7176 [00:00<?, ? examples/s]

Map:   0%|          | 0/7176 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['Document', 'Topic_group', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 33485
    })
    validation: Dataset({
        features: ['Document', 'Topic_group', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 7176
    })
    test: Dataset({
        features: ['Document', 'Topic_group', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 7176
    })
})

In [ ]:
# Keep only model inputs + label
cols_to_keep = ["input_ids", "attention_mask", "label"]
cols_to_remove = [c for c in ds_tok["train"].column_names if c not in cols_to_keep]

ds_tok = ds_tok.remove_columns(cols_to_remove)
ds_tok.set_format(type="torch")
ds_tok


DatasetDict({
    train: Dataset({
        features: ['label', 'input_ids', 'attention_mask'],
        num_rows: 33485
    })
    validation: Dataset({
        features: ['label', 'input_ids', 'attention_mask'],
        num_rows: 7176
    })
    test: Dataset({
        features: ['label', 'input_ids', 'attention_mask'],
        num_rows: 7176
    })
})

In [ ]:
total_params = sum(p.numel() for p in model_student.parameters())
print(f"Total parameters: {total_params:,}")

Total parameters: 66,959,624


In [ ]:
dtypes = {p.dtype for p in model_student.parameters()}
dtypes

{torch.float32}

In [ ]:
model_student.eval()
num_labels = model_student.config.num_labels
num_labels

8

In [ ]:
tokenizer_student

BertTokenizerFast(name_or_path='saketgarodia1/bert-it-ticket-student', vocab_size=30522, model_max_length=512, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, clean_up_tokenization_spaces=False, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}
)

In [ ]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer_student)

val_loader = DataLoader(
    ds_tok["validation"],
    batch_size=VAL_BATCH_SIZE,
    shuffle=False,
    collate_fn=data_collator,
)

test_loader = DataLoader(
    ds_tok["test"],
    batch_size=TEST_BATCH_SIZE,
    shuffle=False,
    collate_fn=data_collator,
)


In [ ]:
# for batch in val_loader:
#     print(batch.keys())

In [ ]:
ds_tok

DatasetDict({
    train: Dataset({
        features: ['label', 'input_ids', 'attention_mask'],
        num_rows: 33485
    })
    validation: Dataset({
        features: ['label', 'input_ids', 'attention_mask'],
        num_rows: 7176
    })
    test: Dataset({
        features: ['label', 'input_ids', 'attention_mask'],
        num_rows: 7176
    })
})

Evaluation helper (accuracy + macro F1)


In [ ]:
def evaluate_model(
    model: torch.nn.Module,
    dataloader: DataLoader,
) -> Dict[str, float]:
    """
    Evaluate a classification model on a DataLoader.
    Returns:
        {"accuracy": float, "f1_macro": float, "loss": float}
    """
    model.eval()
    all_labels = []
    all_preds = []
    losses = []

    # Loss for reporting (standard CE on hard labels)
    criterion = torch.nn.CrossEntropyLoss()

    # Infer device from model
    model_device = next(model.parameters()).device

    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch["input_ids"].to(model_device)
            attention_mask = batch["attention_mask"].to(model_device)
            labels = batch["labels"].to(model_device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
            )
            logits = outputs.logits

            loss = criterion(logits, labels)
            losses.append(loss.item())

            preds = logits.argmax(dim=-1)
            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())

    acc = accuracy_score(all_labels, all_preds)
    f1_macro = f1_score(all_labels, all_preds, average="macro")

    return {
        "loss": sum(losses) / len(losses),
        "accuracy": acc,
        "f1_macro": f1_macro,
    }


Helper: inspect dtype & estimated memory usage


In [ ]:
from collections import defaultdict
from typing import Dict, Tuple

def inspect_model_dtype_and_size_mixed(model: torch.nn.Module) -> Tuple[Dict[torch.dtype, int], int]:
    """
    Inspect a model that may have **mixed dtypes** (e.g. FP16 + INT8 from bitsandbytes).

    Prints:
      - parameter counts per dtype
      - total number of parameters
      - exact parameter memory usage in MB (summing real tensor sizes)

    Returns:
      (params_per_dtype, total_params)
        where params_per_dtype: {dtype -> count_of_parameters}
    """
    params_per_dtype = defaultdict(int)
    total_bytes = 0

    for p in model.parameters():
        if p is None:
            continue
        dt = p.dtype
        n = p.numel()
        params_per_dtype[dt] += n

        # element_size() = bytes per element for this dtype
        total_bytes += n * p.element_size()

    total_params = sum(params_per_dtype.values())
    size_mb = total_bytes / (1024 ** 2)

    print("Parameter dtypes and counts:")
    for dt, n in params_per_dtype.items():
        print(f"  {dt}: {n:,} params")

    print(f"\nTotal parameters: {total_params:,}")
    print(f"Approx. parameter memory (all dtypes combined): {size_mb:.2f} MB\n")

    return params_per_dtype, total_params


In [ ]:
inspect_model_dtype_and_size_mixed(model_student)

Parameter dtypes and counts:
  torch.float32: 66,959,624 params

Total parameters: 66,959,624
Approx. parameter memory (all dtypes combined): 255.43 MB



(defaultdict(int, {torch.float32: 66959624}), 66959624)

In [ ]:
student_fp32 = model_student
student_fp32

DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


In [ ]:
val_loader

In [ ]:
metrics_val_fp32 = evaluate_model(student_fp32, val_loader)
metrics_test_fp32 = evaluate_model(student_fp32, test_loader)

print("FP32 student – Validation:", metrics_val_fp32)
print("FP32 student – Test:", metrics_test_fp32)


FP32 student – Validation: {'loss': 0.3955650680098269, 'accuracy': 0.8851727982162765, 'f1_macro': 0.8848098623203573}
FP32 student – Test: {'loss': 0.4296033374696142, 'accuracy': 0.8786231884057971, 'f1_macro': 0.8755284273181243}


FP16 via PyTorch: `model.half()`
#
# We cast the FP32 student to FP16 using pure PyTorch and re-evaluate.


In [ ]:
student_half_torch = AutoModelForSequenceClassification.from_pretrained(
    STUDENT_MODEL_ID
)
student_half_torch.to(device)

# Cast all params & buffers to float16
student_half_torch.half()

inspect_model_dtype_and_size_mixed(student_half_torch)


Parameter dtypes and counts:
  torch.float16: 66,959,624 params

Total parameters: 66,959,624
Approx. parameter memory (all dtypes combined): 127.72 MB



(defaultdict(int, {torch.float16: 66959624}), 66959624)

In [ ]:
metrics_val_half_torch = evaluate_model(student_half_torch, val_loader)
metrics_test_half_torch = evaluate_model(student_half_torch, test_loader)

print("FP16 (PyTorch .half) – Validation:", metrics_val_half_torch)
print("FP16 (PyTorch .half) – Test:", metrics_test_half_torch)


FP16 (PyTorch .half) – Validation: {'loss': 0.39551276312934025, 'accuracy': 0.8851727982162765, 'f1_macro': 0.8848098623203573}
FP16 (PyTorch .half) – Test: {'loss': 0.4295584954155816, 'accuracy': 0.8784838350055741, 'f1_macro': 0.87544072134907}


FP16 via Hugging Face: `torch_dtype=torch.float16`
#
# Same idea, but we ask `from_pretrained` to load directly in half precision.


In [ ]:
student_half_hf = AutoModelForSequenceClassification.from_pretrained(
    STUDENT_MODEL_ID,
    torch_dtype=torch.float16,   # HF casts as it builds the model
)
student_half_hf.to(device)

inspect_model_dtype_and_size_mixed(student_half_hf)


Parameter dtypes and counts:
  torch.float16: 66,959,624 params

Total parameters: 66,959,624
Approx. parameter memory (all dtypes combined): 127.72 MB



(defaultdict(int, {torch.float16: 66959624}), 66959624)

In [ ]:
metrics_val_half_hf = evaluate_model(student_half_hf, val_loader)
metrics_test_half_hf = evaluate_model(student_half_hf, test_loader)

print("FP16 (HF torch_dtype) – Validation:", metrics_val_half_hf)
print("FP16 (HF torch_dtype) – Test:", metrics_test_half_hf)


FP16 (HF torch_dtype) – Validation: {'loss': 0.39551276312934025, 'accuracy': 0.8851727982162765, 'f1_macro': 0.8848098623203573}
FP16 (HF torch_dtype) – Test: {'loss': 0.4295584954155816, 'accuracy': 0.8784838350055741, 'f1_macro': 0.87544072134907}


 4-bit quantization with BitsAndBytes (NF4)
#
# We'll:
# 1. Define a BitsAndBytesConfig for 4-bit NF4.
# 2. Load the student with `quantization_config=nf4_config`.
# 3. Evaluate on the validation + test sets.


bitsandbytes compresses transformer weights into very small formats (like 4-bit NF4) while still doing all computations in higher precision (fp16/bf16) to preserve accuracy. When you load a model using `load_in_4bit=True`, you are telling Hugging Face to *store* all eligible model weights in 4-bit form instead of fp16/fp32, immediately reducing memory by 4×–8×. The argument `bnb_4bit_quant_type="nf4"` specifies the quantization method: NF4 (NormalFloat4) is the best-performing 4-bit format because it places quantization buckets according to the normal distribution of real transformer weights, giving much lower error than simple uniform 4-bit quantization.

During inference, bitsandbytes must *dequantize* these 4-bit weights back into a higher-precision format before multiplying them with activations. That behavior is controlled by `bnb_4bit_compute_dtype=torch.bfloat16`, which tells bitsandbytes to perform matmuls using bf16 — a numerically stable 16-bit format that avoids many fp16 issues. Even though storage is only 4 bits, the math itself happens in bf16 to keep accuracy high. The argument `bnb_4bit_use_double_quant=True` enables a second layer of quantization applied to the *scales* used during NF4 quantization, giving extra memory savings with almost no quality loss.

Putting all of these together, when you load a model with `quantization_config=nf4_config`, Hugging Face will replace standard Linear layers with 4-bit bitsandbytes layers, quantize the downloaded weights into NF4 blocks, optionally quantize the associated scales (double quantization), and compute all forward passes in bf16. The end result is a compact, GPU-friendly, 4-bit model with strong accuracy retention — ideal for running large models on small hardware.


In [ ]:
from transformers import BitsAndBytesConfig

nf4_config = BitsAndBytesConfig(
    load_in_4bit=True,                # use 4-bit weights
    bnb_4bit_quant_type="nf4",        # NormalFloat4
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

student_4bit = AutoModelForSequenceClassification.from_pretrained(
    STUDENT_MODEL_ID,
    quantization_config=nf4_config,
    device_map="auto",               # let HF place modules on GPU
)

# Inspect (dtype set will look different; params are handled by bitsandbytes)
inspect_model_dtype_and_size_mixed(student_4bit)


Parameter dtypes and counts:
  torch.float16: 23,902,472 params
  torch.uint8: 21,528,576 params

Total parameters: 45,431,048
Approx. parameter memory (all dtypes combined): 66.12 MB



(defaultdict(int, {torch.float16: 23902472, torch.uint8: 21528576}), 45431048)

In [ ]:
for name, p in student_4bit.named_parameters():
    print(f"{name:60} {str(p.dtype):12} {p.numel():10}")


distilbert.embeddings.word_embeddings.weight                 torch.float16   23440896
distilbert.embeddings.position_embeddings.weight             torch.float16     393216
distilbert.embeddings.LayerNorm.weight                       torch.float16        768
distilbert.embeddings.LayerNorm.bias                         torch.float16        768
distilbert.transformer.layer.0.attention.q_lin.weight        torch.uint8      294912
distilbert.transformer.layer.0.attention.q_lin.bias          torch.float16        768
distilbert.transformer.layer.0.attention.k_lin.weight        torch.uint8      294912
distilbert.transformer.layer.0.attention.k_lin.bias          torch.float16        768
distilbert.transformer.layer.0.attention.v_lin.weight        torch.uint8      294912
distilbert.transformer.layer.0.attention.v_lin.bias          torch.float16        768
distilbert.transformer.layer.0.attention.out_lin.weight      torch.uint8      294912
distilbert.transformer.layer.0.attention.out_lin.bias     

# ✅ 1. Why Embeddings Are **NOT** Quantized

Embeddings are the largest matrix in the model—but also the most fragile.

### 🔍 What do embeddings do?

For a token `"network"`, the embedding table does:


Where **E** is a huge matrix, e.g.:

- **DistilBERT:** `30522 × 768` ≈ **23.4M parameters**

Each row is a semantic vector.  
If distorted, meaning collapses → the model cannot interpret text.

---

### ❌ Why embeddings break when quantized

Example FP16 embedding for `"server"`:


Quantizing to **4-bit (NF4)** forces numbers into ~16 values, becoming:


This destroys:

- synonym relationships  
- semantic topology  
- positional structure  

➡️ The **embedding space collapses**.

### 📉 Real-world effect

Quantizing embeddings causes **20–40% accuracy loss**.  
Models literally cannot parse meaning.

### 📌 Embeddings = semantic core → **must stay FP16/BF16**

All practical systems keep embeddings in higher precision:

- LLaMA  
- GPTQ  
- BitsAndBytes NF4  
- AWQ  
- QLoRA  
- GGUF / llama.cpp  

---

# ✅ 2. Why Biases Are **NOT** Quantized

Bias vectors are tiny (size ≈ hidden dimension).

Example (DistilBERT, hidden size = 768):

- Weight matrix `W`: `768 × 768` → **590K params** (quantized)
- Bias vector `b`: `768` → **768 params** (kept FP16)

### Why keep biases FP16?

#### ✔ They control the *offset*, not the scale  
Quantization errors here shift activations too much.

#### ✔ They are tiny  
Quantizing bias saves almost nothing:


But risks **breaking layer stability**.

#### ✔ Bias affects pre-activation distribution


If `b` is rounded aggressively (4-bit):

- output distribution shifts  
- GELU/ReLU behave incorrectly  
- gradients destabilize  

📌 **Small memory cost → huge stability impact.**  
Bias stays FP16.

---

# ✅ 3. Why LayerNorm Parameters Are **NOT** Quantized

LayerNorm has two vectors:

- **gamma** (scale) – `hidden_dim`
- **beta** (shift) – `hidden_dim`

Also tiny in size.

### 🔍 Why quantizing LayerNorm destroys models

LayerNorm does:


This small error is applied:

- at **every token**
- in **every layer**
- through the whole transformer

📌 Errors accumulate catastrophically → model destabilizes.

---

# 🔥 Summary (Easy Version)

| Component          | Quantize? | Why |
|-------------------|-----------|-----|
| **Embeddings**    | ❌ No     | Destroy semantic meaning; massive accuracy drop |
| **Biases**        | ❌ No     | Tiny but sensitive; shift distributions |
| **LayerNorm γ, β**| ❌ No     | Critical for stability; tiny size |
| **Attention weights** | ✅ Yes | Large matrices; quantization-safe |
| **FFN weights**   | ✅ Yes    | Largest weights; tolerate quantization |
| **Final classifier** | Often FP16 | Needs clean logits for accuracy |

---

# 🎯 Intuition Summary

- 🧠 **Embeddings** = language understanding → must stay precise  
- ⚙️ **Bias + LayerNorm** = stability controls → quantizing breaks the model  
- 🏋️ **FFN & Attention weights** = heavy lifting → can be quantized safely  

---

# 🎵 Analogy

Think of quantizing a song:

- **Embeddings = the vocalist’s voice** → must stay clear  
- **LayerNorm + biases = EQ knobs** → tiny changes ruin the mix  
- **FFN weights = background instruments** → can be compressed heavily  



In [ ]:
metrics_val_4bit = evaluate_model(student_4bit, val_loader)
metrics_test_4bit = evaluate_model(student_4bit, test_loader)

print("4-bit NF4 student – Validation:", metrics_val_4bit)
print("4-bit NF4 student – Test:", metrics_test_4bit)


4-bit NF4 student – Validation: {'loss': 0.38936258951822916, 'accuracy': 0.8843366778149386, 'f1_macro': 0.882413885496768}
4-bit NF4 student – Test: {'loss': 0.4223681979709201, 'accuracy': 0.8798773690078038, 'f1_macro': 0.8754891651185079}


Push 4-bit student to Hugging Face Hub
#
# This will create a new model repo like:
# `saketgarodia1/distilbert-IT-ticket-student-4bit`


In [ ]:
from huggingface_hub import notebook_login

notebook_login()   # run once per session and paste your HF token


In [ ]:
FOUR_BIT_REPO_ID = "saketgarodia1/distilbert-IT-ticket-student-4bit"

# Make sure tokenizer is the same one you've been using
student_4bit.push_to_hub(FOUR_BIT_REPO_ID)
tokenizer_student.push_to_hub(FOUR_BIT_REPO_ID)


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...za868l5/model.safetensors:   1%|          |  525kB / 70.1MB            

README.md: 0.00B [00:00, ?B/s]

CommitInfo(commit_url='https://huggingface.co/saketgarodia1/distilbert-IT-ticket-student-4bit/commit/de19d84ea91706e882d4830c8f49bc2f47258c1f', commit_message='Upload tokenizer', commit_description='', oid='de19d84ea91706e882d4830c8f49bc2f47258c1f', pr_url=None, repo_url=RepoUrl('https://huggingface.co/saketgarodia1/distilbert-IT-ticket-student-4bit', endpoint='https://huggingface.co', repo_type='model', repo_id='saketgarodia1/distilbert-IT-ticket-student-4bit'), pr_revision=None, pr_num=None)